# GAE_HEP on Kaggle GPU

Permutation-invariant graph autoencoders for unsupervised anomaly detection on jets of the LHC Olympics 2020 R&D dataset ([github.com/raktim711/GAE_HEP](https://github.com/raktim711/GAE_HEP)).

This notebook runs the full pipeline at GPU scale: it downloads and preprocesses the data (Zenodo record 6466204, about 3.2 GB), then trains the set autoencoder (3 seeds), its mass-conditional variant, the node-level GAE, Deep SVDD and the supervised classifier reference.

Every run trains, scores and evaluates through the `gaehep` CLI; the latent-flow, quantile-normalised and ensemble variants and the summary report follow, and the results are zipped for download at the end.

> **Before you run:** in the Kaggle session options (right-hand panel) switch **Internet on** (needed for `git clone`, `pip install` and the Zenodo download) and select a **GPU accelerator** under *Accelerator*. Training uses a single GPU (`train.device=cuda`).

In [ ]:
!nvidia-smi

import os

REPO_URL = "https://github.com/raktim711/GAE_HEP.git"
REPO_DIR = "/kaggle/working/GAE_HEP"

# clone once; skipped when the repo is already there (e.g. after a kernel restart)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

import torch

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible - choose a GPU accelerator in the session options.")

In [ ]:
import os

# Kaggle: keep everything under /kaggle/working
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"

# data pipeline; the shell commands inherit os.environ.
# `download` is md5-verified and resumable: re-run this cell if it gets interrupted.
!gaehep download
!gaehep preprocess --sample rnd --workers 4
!gaehep preprocess --sample qqq --workers 4
!gaehep tables --sample rnd
!gaehep tables --sample qqq
!gaehep splits
!gaehep baselines

## GPU-scale training runs

The YAML files in `configs/` are sized for a CPU machine (top-50 constituents, 48-wide EdgeConv blocks, 200k jets per epoch, batch size 256). Here every model is trained with

`gaehep train configs/<config>.yaml --name <name> --set key=value ...`

and the overrides below; anything not listed (learning rate, weight decay, patience, latent size, ...) keeps its config value.

| override | applies to | config default | GPU run |
|---|---|---|---|
| `train.device` | all | not set | `cuda` |
| `train.batch_size` | all | 256 | 1024 |
| `train.jets_per_epoch` | all | 200000 | 1000000 |
| `train.epochs` | all | 25 (classifier: 20) | 40 |
| `train.time_budget_min` | all | 100 | 480 |
| `data.n_const` | all | 50 | 100 (all stored constituents) |
| `model_args.k` | all | 12 | 16 |
| `model_args.hidden` | set_ae, set_ae_cond, deep_svdd, classifier | [48, 48, 48] | [128,128,128] |
| `model_args.n_out` | set_ae, set_ae_cond | 50 | 100 (must equal `data.n_const`) |
| `model_args.enc_hidden`, `model_args.dec_hidden` | node_gae | [48, 48] each | [128,128] each |

**Seven training runs:** `set_ae` with three seeds (`set_ae_gpu_s1`, `_s2`, `_s3`, combined later in a rank ensemble), `set_ae_cond_gpu`, `node_gae_gpu`, `deep_svdd_gpu` and `classifier_gpu`. Each `run(...)` call does the full CLI cycle **train -> score -> evaluate** and writes to `results/runs/<name>/`.

**Resumable:** training writes checkpoints, and `train.time_budget_min=480` caps a run at 8 h so it can stop cleanly inside Kaggle's 9 h session limit. If the session limit hits or the kernel is interrupted, re-run the same cell and unfinished runs continue from their last checkpoint (this needs the files in `/kaggle/working` to still exist, e.g. after a kernel restart or when the saved output is restored).

**YAML lists** must be passed exactly like `model_args.hidden=[128,128,128]`, without spaces, because each override is a single `--set` argument.

In [ ]:
import os
import shlex
import subprocess
import time

REPO_DIR = "/kaggle/working/GAE_HEP"

# set again so that this cell (and the next ones) also work after a kernel restart
os.environ["GAEHEP_DATA"] = "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"
os.chdir(REPO_DIR)


def run(config, name, *overrides):
    """Run `gaehep train configs/<config>.yaml --name <name> --set k=v ...` (train -> score -> evaluate).

    stdout/stderr are not captured, so the training log streams live into the cell output.
    A non-zero exit status raises CalledProcessError and stops the cell (check=True).
    """
    cmd = ["gaehep", "train", f"configs/{config}.yaml", "--name", name]
    for override in overrides:
        cmd += ["--set", override]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    subprocess.run(
        cmd,
        check=True,
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},  # no block-buffering of the child's prints
    )
    print(f"[{name}] wall time: {(time.time() - t0) / 60:.1f} min", flush=True)


# overrides shared by every model
GPU = [
    "train.device=cuda",
    "train.batch_size=1024",  # lower to 512 if you hit a CUDA out-of-memory error
    "train.jets_per_epoch=1000000",
    "train.epochs=40",
    "train.time_budget_min=480",
    "data.n_const=100",
    "model_args.k=16",
]

# model-specific widths (YAML lists: no spaces!)
SET_AE = GPU + ["model_args.n_out=100", "model_args.hidden=[128,128,128]"]  # set_ae, set_ae_cond
NODE_GAE = GPU + ["model_args.enc_hidden=[128,128]", "model_args.dec_hidden=[128,128]"]
DEEP_SVDD = GPU + ["model_args.hidden=[128,128,128]"]
CLASSIFIER = GPU + ["model_args.hidden=[128,128,128]"]

In [ ]:
# ---- seven training runs (train -> score -> evaluate); comment out finished ones if you like ----
run("set_ae", "set_ae_gpu_s1", *SET_AE, "train.seed=1")
run("set_ae", "set_ae_gpu_s2", *SET_AE, "train.seed=2")
run("set_ae", "set_ae_gpu_s3", *SET_AE, "train.seed=3")
run("set_ae_cond", "set_ae_cond_gpu", *SET_AE)
run("node_gae", "node_gae_gpu", *NODE_GAE)
run("deep_svdd", "deep_svdd_gpu", *DEEP_SVDD)
run("classifier", "classifier_gpu", *CLASSIFIER)

# ---- post-hoc variants and the report ----
!gaehep flow set_ae_gpu_s1
!gaehep qn set_ae_gpu_s1
!gaehep ensemble set_ae_gpu_s1 set_ae_gpu_s2 set_ae_gpu_s3 --out set_ae_gpu_ens
!gaehep report

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, Markdown, display

RESULTS = Path(os.environ.get("GAEHEP_RESULTS", "/kaggle/working/results"))
ZIP_PATH = Path("/kaggle/working/gaehep_results.zip")

# 1) the summary table written by `gaehep report`
summary = RESULTS / "summary.md"
if summary.is_file():
    display(Markdown(summary.read_text()))
else:
    print(f"{summary} not found - run the previous cell (it ends with `gaehep report`) first.")


# 2) zip the results for download, leaving out model weights / checkpoints
def is_checkpoint(path: Path) -> bool:
    parts = [p.lower() for p in path.relative_to(RESULTS).parts]
    return path.suffix.lower() in {".pt", ".pth", ".ckpt"} or any(
        "checkpoint" in p or p.startswith("ckpt") for p in parts
    )


added = skipped = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RESULTS.rglob("*")):
        if not path.is_file():
            continue
        if is_checkpoint(path):
            skipped += 1
            continue
        zf.write(path, arcname=str(Path("results") / path.relative_to(RESULTS)))
        added += 1

print(f"{ZIP_PATH}: {added} files, {ZIP_PATH.stat().st_size / 1e6:.1f} MB "
      f"({skipped} *.pt / checkpoint files left out)")

# download link (the file also shows up in the notebook's Output tab)
cwd = os.getcwd()
try:
    os.chdir(ZIP_PATH.parent)
    display(FileLink(ZIP_PATH.name))
finally:
    os.chdir(cwd)